In [3]:
from sklearn.cluster import KMeans
from blazingsql import BlazingContext
import matplotlib.pyplot as plt
import seaborn as sns
import cudf
import time

bc_cudf = BlazingContext()

BlazingContext ready


In [ ]:
netflow_gdf_cudf = cudf.read_csv('./data/nf-chunk2.csv')
bc_cudf.create_table('netflow', netflow_gdf_cudf)

In [ ]:
query = '''
        SELECT
            a.firstSeenSrcTotalBytes as bytesOut,
            a.firstSeenDestTotalBytes as bytesIn
        FROM
            netflow a
        '''

%prun result = bc_cudf.sql(query)

In [ ]:
times=[]

for i in range(3, 5):
    t0 = time.time()
    %prun kmeans = KMeans(n_clusters=i,algorithm='auto').fit(result)
    t1 = time.time()
    times.append(t1-t0)

    result['cluster_id'] = kmeans.labels_
    sns.lmplot('bytesOut', 'bytesIn', data=result, fit_reg=False,
               scatter_kws={"s":150}, hue="cluster_id")

    plt.title('kmean plot' + i)
    plt.show()

In [ ]:
print(times)